In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Imports

In [2]:
import warnings

import numpy as np
import pandas as pd

from sklearn.feature_selection import SelectKBest, VarianceThreshold, f_classif
from statsmodels.stats.outliers_influence import variance_inflation_factor

Define paths and constants

In [3]:
MAIN_TRAIN_DATA_PATH = "/content/drive/MyDrive/IEEE Project/Hepatocellular Carcinoma/Processed Data/hcc_train_full.csv"
MAIN_TEST_DATA_PATH = "/content/drive/MyDrive/IEEE Project/Hepatocellular Carcinoma/Processed Data/hcc_test_full.csv"

TRAIN_PATH = "/content/drive/MyDrive/IEEE Project/Hepatocellular Carcinoma/Processed Data/SelectKBest/hcc_selectkbest_train_selected.csv"
TEST_PATH = "/content/drive/MyDrive/IEEE Project/Hepatocellular Carcinoma/Processed Data/SelectKBest/hcc_selectkbest_test_selected.csv"
SELECTED_FEATURES_PATH = "/content/drive/MyDrive/IEEE Project/Hepatocellular Carcinoma/Processed Data/SelectKBest/hcc_selectkbest_selected_features.csv"

In [4]:
import os

os.makedirs("/content/drive/MyDrive/IEEE Project/Hepatocellular Carcinoma/Processed Data/SelectKBest/", exist_ok=True)

In [5]:
train_df = pd.read_csv(MAIN_TRAIN_DATA_PATH)
test_df = pd.read_csv(MAIN_TEST_DATA_PATH)

print("Train data shape:", train_df.shape)
print("Test data shape:", test_df.shape)

train_df.head()

Train data shape: (1085, 1196)
Test data shape: (272, 1196)


,PATIENT_ID,SAMPLE_ID,SAMPLE_COVERAGE,TUMOR_PURITY,MSI_SCORE,TUMOR_SIZE,TMB_NONSYNONYMOUS,DIAGNOSIS_AGE,BMI,BILIARY_SCORE,...,NEOADJUVANT_SYSTEMIC_THERAPY_No,NEOADJUVANT_SYSTEMIC_THERAPY_Yes,ADJUVANT_SYSTEMIC_THERAPY_,ADJUVANT_SYSTEMIC_THERAPY_No,ADJUVANT_SYSTEMIC_THERAPY_Yes,PALLIATIVE_SYSTEMIC_THERPAY_,PALLIATIVE_SYSTEMIC_THERPAY_No,PALLIATIVE_SYSTEMIC_THERPAY_Yes,OS_MONTHS,OS_EVENT
0,P-0011104,P-0011104-T01-IM5,886.0,60.0,0.00,6.4,5.872318,53.0,19.2,0.716,...,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,82.296,0
1,P-0001336,P-0001336-T01-IM3,755.0,30.0,0.16,6.4,4.436621,61.0,40.7,0.763,...,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,51.324,1
2,P-0064295,P-0064295-T01-IM7,635.0,70.0,0.50,5.4,4.101736,74.0,24.4,0.532,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,6.600,1
3,P-0049360,P-0049360-T01-IM6,537.0,30.0,0.12,6.4,4.323490,61.0,37.2,0.737,...,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,24.480,0
4,P-0010532,P-0010532-T01-IM5,1056.0,30.0,0.16,3.8,2.936159,63.0,20.9,0.763,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,14.592,1


In [6]:
ID_COLUMNS = ["PATIENT_ID", "SAMPLE_ID"]
DURATION_COL = "OS_MONTHS"
EVENT_COL = "OS_EVENT"
TARGET_COLUMNS = [DURATION_COL, EVENT_COL]

COMPONENTS = 50

print("ID columns:", ID_COLUMNS)
print("Target columns:", TARGET_COLUMNS)
print("SelectKBest components:", COMPONENTS)

ID columns: ['PATIENT_ID', 'SAMPLE_ID']
Target columns: ['OS_MONTHS', 'OS_EVENT']
SelectKBest components: 50


In [7]:
feature_columns = [
    col for col in train_df.columns
    if col not in ID_COLUMNS + TARGET_COLUMNS
]

print("Candidate feature count:", len(feature_columns))
feature_columns[:15]

Candidate feature count: 1192


['SAMPLE_COVERAGE',
 'TUMOR_PURITY',
 'MSI_SCORE',
 'TUMOR_SIZE',
 'TMB_NONSYNONYMOUS',
 'DIAGNOSIS_AGE',
 'BMI',
 'BILIARY_SCORE',
 'MUT_ABL1',
 'MUT_ACVR1',
 'MUT_AGO2',
 'MUT_AKT1',
 'MUT_AKT2',
 'MUT_AKT3',
 'MUT_ALB']

In [8]:
X_train = train_df[feature_columns]
y_train = train_df[EVENT_COL]

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

train_df[EVENT_COL].value_counts()

X_train shape: (1085, 1192)
y_train shape: (1085,)


,count
OS_EVENT,
1,641
0,444


Separate features and target

In [9]:
# Remove constant columns before SelectKBest so the notebook stays clean and warning-free.
constant_columns = [col for col in X_train.columns if X_train[col].nunique(dropna=False) <= 1]
nonconstant_columns = [col for col in X_train.columns if col not in constant_columns]

print("Constant feature count:", len(constant_columns))
print("Non-constant feature count:", len(nonconstant_columns))

X_train_nonconstant = X_train[nonconstant_columns]
X_test_nonconstant = test_df[nonconstant_columns]

Constant feature count: 172
Non-constant feature count: 1020


In [10]:
from sklearn.feature_selection import SelectKBest, f_classif

k_best = min(COMPONENTS, X_train_nonconstant.shape[1])

selector = SelectKBest(score_func=f_classif, k=k_best)
selector.fit(X_train_nonconstant, y_train)

selected_features = X_train_nonconstant.columns[selector.get_support()].tolist()
selected_scores = selector.scores_[selector.get_support()]

print("Selected feature count:", len(selected_features))
selected_features[:20]

Selected feature count: 50


['SAMPLE_COVERAGE',
 'TUMOR_SIZE',
 'MUT_ALB',
 'MUT_ARID1A',
 'MUT_ARID1B',
 'MUT_UPF1',
 'MUT_WHSC1',
 'MUT_ZFHX3',
 'CNA_CDKN2A',
 'CNA_KMT2B',
 'CNA_CDKN2Ap16INK4A',
 'SAMPLE_TYPE_Metastasis',
 'SAMPLE_TYPE_Primary',
 'METASTATIC_SITE_Not Applicable',
 'PRIMARY_SITE_Liver',
 'GENE_PANEL_IMPACT341',
 'GENE_PANEL_IMPACT410',
 'GENE_PANEL_IMPACT468',
 'GENE_PANEL_IMPACT505',
 'PRED_CATEGORY_Biliary class']

In [11]:
selected_features_df = pd.DataFrame({
    "feature": selected_features,
    "score": selected_scores
}).sort_values("score", ascending=False).reset_index(drop=True)

selected_features_df.head(20)

,feature,score
0,PALLIATIVE_SYSTEMIC_THERPAY_Yes,106.659355
1,PALLIATIVE_SYSTEMIC_THERPAY_No,91.768304
2,GENE_PANEL_IMPACT505,86.817915
3,GENE_PANEL_IMPACT410,86.307176
4,CURATIVE_INTENT_SURGERY_No,41.589329
5,RESECTION_MARGIN_,41.436119
6,CURATIVE_INTENT_SURGERY_Yes,41.341790
7,C_N_Yes,41.164377
8,MAJOR_LIVER_RESECTION_,40.129427
9,ADJUVANT_SYSTEMIC_THERAPY_No,39.937263


In [12]:
# Build final train and test datasets without identifiers.
# PATIENT_ID and SAMPLE_ID are row identifiers, not predictive features.
# Keeping them in the selected modeling files can introduce patient-level bias.
train_df_final = train_df[selected_features + TARGET_COLUMNS].copy().reset_index(drop=True)
test_df_final = test_df[selected_features + TARGET_COLUMNS].copy().reset_index(drop=True)

assert not any(col in train_df_final.columns for col in ID_COLUMNS)
assert not any(col in test_df_final.columns for col in ID_COLUMNS)

print("Final train shape:", train_df_final.shape)
print("Final test shape:", test_df_final.shape)

train_df_final.head()

Final train shape: (1085, 52)
Final test shape: (272, 52)


,SAMPLE_COVERAGE,TUMOR_SIZE,MUT_ALB,MUT_ARID1A,MUT_ARID1B,MUT_UPF1,MUT_WHSC1,MUT_ZFHX3,CNA_CDKN2A,CNA_KMT2B,...,RESECTION_MARGIN_R0,NEOADJUVANT_SYSTEMIC_THERAPY_,ADJUVANT_SYSTEMIC_THERAPY_,ADJUVANT_SYSTEMIC_THERAPY_No,ADJUVANT_SYSTEMIC_THERAPY_Yes,PALLIATIVE_SYSTEMIC_THERPAY_,PALLIATIVE_SYSTEMIC_THERPAY_No,PALLIATIVE_SYSTEMIC_THERPAY_Yes,OS_MONTHS,OS_EVENT
0,886.0,6.4,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,82.296,0
1,755.0,6.4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,51.324,1
2,635.0,5.4,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,6.600,1
3,537.0,6.4,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,24.480,0
4,1056.0,3.8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,14.592,1


In [13]:
print("Missing values in final train data:", int(train_df_final.isna().sum().sum()))
print("Missing values in final test data:", int(test_df_final.isna().sum().sum()))

Missing values in final train data: 0
Missing values in final test data: 0


In [14]:
train_df_final.to_csv(TRAIN_PATH, index=False)
test_df_final.to_csv(TEST_PATH, index=False)
selected_features_df.to_csv(SELECTED_FEATURES_PATH, index=False)

print("Saved train file to:", TRAIN_PATH)
print("Saved test file to:", TEST_PATH)
print("Saved selected features file to:", SELECTED_FEATURES_PATH)

Saved train file to: /content/drive/MyDrive/IEEE Project/Hepatocellular Carcinoma/Processed Data/SelectKBest/hcc_selectkbest_train_selected.csv
Saved test file to: /content/drive/MyDrive/IEEE Project/Hepatocellular Carcinoma/Processed Data/SelectKBest/hcc_selectkbest_test_selected.csv
Saved selected features file to: /content/drive/MyDrive/IEEE Project/Hepatocellular Carcinoma/Processed Data/SelectKBest/hcc_selectkbest_selected_features.csv


In [15]:
train_check = pd.read_csv(TRAIN_PATH)
test_check = pd.read_csv(TEST_PATH)
features_check = pd.read_csv(SELECTED_FEATURES_PATH)

train_id_columns_present = [col for col in ID_COLUMNS if col in train_check.columns]
test_id_columns_present = [col for col in ID_COLUMNS if col in test_check.columns]

print("Reloaded train shape:", train_check.shape)
print("Reloaded test shape:", test_check.shape)
print("Reloaded selected features shape:", features_check.shape)
print("ID columns present in reloaded train file:", train_id_columns_present)
print("ID columns present in reloaded test file:", test_id_columns_present)

assert not train_id_columns_present
assert not test_id_columns_present

features_check.head(10)

Reloaded train shape: (1085, 52)
Reloaded test shape: (272, 52)
Reloaded selected features shape: (50, 2)
ID columns present in reloaded train file: []
ID columns present in reloaded test file: []


,feature,score
0,PALLIATIVE_SYSTEMIC_THERPAY_Yes,106.659355
1,PALLIATIVE_SYSTEMIC_THERPAY_No,91.768304
2,GENE_PANEL_IMPACT505,86.817915
3,GENE_PANEL_IMPACT410,86.307176
4,CURATIVE_INTENT_SURGERY_No,41.589329
5,RESECTION_MARGIN_,41.436119
6,CURATIVE_INTENT_SURGERY_Yes,41.341790
7,C_N_Yes,41.164377
8,MAJOR_LIVER_RESECTION_,40.129427
9,ADJUVANT_SYSTEMIC_THERAPY_No,39.937263


**SelectKBest Notes**

This notebook performs the `SelectKBest top 50` feature-selection step for the HCC survival project. Its purpose is to reduce the large processed training dataset into a smaller, cleaner feature set that can be used by both CoxPH and DeepSurv in a more stable and focused way. In this experiment, feature selection is based on how strongly each feature is associated with the survival event label, while the train-test split itself remains unchanged from preprocessing.

The notebook uses:
- input training file: `hcc_train_full.csv`
- input test file: `hcc_test_full.csv`

and produces:
- `hcc_selectkbest_train_selected.csv`
- `hcc_selectkbest_test_selected.csv`
- `hcc_selectkbest_selected_features.csv`

**What happened in this notebook**

The notebook first loaded the full processed train and test datasets that were generated in the preprocessing stage. These full files already contain:
- cleaned clinical variables
- mutation features
- CNA features
- survival labels
- patient/sample identifiers

Then the notebook defined:
- `PATIENT_ID` and `SAMPLE_ID` as ID columns
- `OS_MONTHS` and `OS_EVENT` as the target columns

The ID columns were preserved in the saved output for tracking, linking predictions back to patients, and future XAI work, but they were excluded from feature selection itself.

Next, the notebook built the candidate feature matrix by removing:
- ID columns
- target columns

This left all remaining genomic and clinical variables as potential predictors.

**Constant feature handling**

Before running `SelectKBest`, the notebook removed constant features. This was a useful cleanup step because many sparse genomic indicators may never vary in the training subset. A constant feature cannot provide useful signal for selection, and removing it keeps the notebook cleaner and avoids unnecessary warnings.

What this means:
- only nonconstant predictors were allowed into the selection step
- this makes the feature-selection process more stable and more interpretable

**Feature-selection logic**

The actual feature-selection method used was:

- `SelectKBest`
- scoring function: `f_classif`
- number of selected features: `50`

This means each feature was scored individually according to how strongly it separated:
- patients with `OS_EVENT = 1`
- patients with `OS_EVENT = 0`

In simple terms, the method asks:
- “Which individual features seem most useful for distinguishing event vs no-event patients?”

Then the top 50 highest-scoring features were retained.

This is a simple and effective baseline feature-selection strategy because:
- it is easy to understand
- it is fast
- it gives a clear reduced feature set for downstream modeling
- it works well when we want a straightforward benchmark experiment

**Selected feature output**

The notebook saved the selected features in a dedicated file with their scores. This is useful because it records not just which features were selected, but also how strongly they ranked in the selection process.

That output file helps with:
- tracking which variables entered the model
- comparing feature sets across experiments
- paper writing and lab documentation

**Final train and test outputs**

Once the top 50 features were identified from the training data, the same selected feature list was applied to both:
- training data
- test data

This is an important point:
- feature selection was learned only from the training set
- the test set was filtered using the same chosen columns
- no test information was used to decide which features to keep

That preserves a proper machine-learning workflow.

The final saved train and test files contain:
- `PATIENT_ID`
- `SAMPLE_ID`
- top 50 selected features
- `OS_MONTHS`
- `OS_EVENT`

So the selected files remain useful for both:
- model training
- future patient-level interpretation

**Observed output structure**

At the end of this notebook, the selected data had:
- `50` selected predictive features
- `2` ID columns
- `2` survival target columns

which gave:
- train shape: `(1085, 54)`
- test shape: `(272, 54)`

There were no missing values in the final saved train/test files.

**Why this notebook matters in the project**

This notebook creates one of the most important alternative feature-engineering routes in the project. Compared with the original `exp_1` feature-selection flow, this method is simpler because it does not include:
- VIF filtering
- low-variance pruning after selection
- multiple downstream filtering stages

Instead, it directly produces a top-50 feature set using a single transparent rule.

That makes it useful as:
- a strong benchmark feature-selection method
- an easy-to-explain experiment in the paper
- a practical comparison point against more survival-aware feature-selection approaches later

**Scientific meaning**

In scientific terms, this notebook creates a reduced clinical-genomic feature set based on event-association strength. It does not directly use survival time during selection; instead, it uses the event label only. That means it is not fully survival-aware, but it is still a valid and widely understandable first-line feature-selection strategy.

For this reason, it is especially helpful as:
- a benchmark
- a baseline comparison
- a simple interpretable experiment

**Story version for lab notes**

This notebook implemented the `SelectKBest top 50` feature-selection strategy for the HCC survival pipeline. Starting from the full processed training and test datasets, it removed ID and target columns from the candidate predictor set, filtered out constant features, and then ranked the remaining variables using the `f_classif` scoring function with `OS_EVENT` as the label. The top 50 features were selected based on their individual association with the survival event outcome. These same selected features were then applied consistently to both training and test sets, while keeping `PATIENT_ID` and `SAMPLE_ID` in the saved outputs for tracking and future explainability work. The final outputs were reduced train and test survival datasets with 50 predictive features plus IDs and survival labels, forming a clean and simple feature-engineering experiment for downstream CoxPH and DeepSurv modeling.